## Cell 1 — Check GPU

In [ ]:
import torch

print("CUDA Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError("GPU not available. Set Runtime → Change runtime type → T4 GPU.")


## Cell 2 — Mount Drive (Recommended)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## Cell 3 — Clone Repository

Edit `REPO_URL` below to match your repository.

In [ ]:
REPO_URL = "https://github.com/Somaskandan931/Monosplat.git"

import os
if os.path.isdir("/content/monosplat/.git"):
    print("Repo already cloned — pulling latest changes...")
    !git -C /content/monosplat pull --ff-only
else:
    print("Fresh clone...")
    !git clone {REPO_URL} /content/monosplat
%cd /content/monosplat
!git log --oneline -3  # confirm which commit is running


## Cell 4 — Install Dependencies

In [ ]:
import subprocess, sys, torch

# Install gsplat wheel matched to this runtime's CUDA version
cuda_ver  = torch.version.cuda or ""
cuda_tag  = "cu" + cuda_ver.replace(".", "")
torch_tag = "pt" + "".join(torch.__version__.split("+")[0].split(".")[:2])
index_url = f"https://docs.gsplat.studio/whl/{torch_tag}{cuda_tag}"
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "gsplat", "--extra-index-url", index_url],
    check=True,
)

# Install project requirements
import os
req = "requirements-colab.txt" if os.path.exists("requirements-colab.txt") else "requirements.txt"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", req], check=True)

print("\n✅ Dependencies installed")


## Cell 5 — Upload Package

Upload your `colab_training_package.zip` when the file picker appears.

In [ ]:
from google.colab import files

uploaded = files.upload()  # select colab_training_package.zip


## Cell 6 — Extract Package

Expected structure inside the zip:
```
frames/
sparse_text/
```

In [ ]:
import zipfile
from pathlib import Path

zip_name = next(n for n in uploaded if n.lower().endswith(".zip"))
DATASET  = Path("/content/dataset")
DATASET.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(zip_name) as zf:
    zf.extractall(DATASET)

# Unwrap single nested directory if present
entries = [p for p in DATASET.iterdir()]
if len(entries) == 1 and entries[0].is_dir():
    DATASET = entries[0]

print("Dataset root:", DATASET)
print("Contents:", [p.name for p in DATASET.iterdir()])


## Cell 7 — Verify Dataset

All files should show ✓ before training.

In [ ]:
from pathlib import Path

required = [
    DATASET / "sparse_text" / "cameras.txt",
    DATASET / "sparse_text" / "images.txt",
    DATASET / "sparse_text" / "points3D.txt",
]

for f in required:
    print(f.name, "✓" if f.exists() else "✗ MISSING")

frames_dir = DATASET / "frames"
if frames_dir.exists():
    n = sum(1 for f in frames_dir.iterdir() if f.suffix.lower() in {".jpg", ".jpeg", ".png"})
    print(f"frames/ ✓  ({n} images)")
else:
    print("frames/ ✗ MISSING")


## Cell 8 — Training

Reads all settings from `configs/config.yaml` in the repository. Exports `final.ply` and `final.splat` automatically when done.

In [ ]:
import subprocess, sys, os, threading
from pathlib import Path

REPO_DIR   = Path("/content/monosplat")
OUTPUT_DIR = Path("/content/output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

cmd = [
    sys.executable,
    str(REPO_DIR / "colab" / "train.py"),
    "--sparse", str(DATASET / "sparse_text"),
    "--frames", str(DATASET / "frames"),
    "--output", str(OUTPUT_DIR),
    "--config", str(REPO_DIR / "configs" / "config.yaml"),
]

env = os.environ.copy()
env["PYTHONPATH"]               = f"{REPO_DIR / 'src'}:{REPO_DIR}"
env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,max_split_size_mb:128"

# ── Live output streaming ────────────────────────────────────────────────────
# subprocess.run() blocks silently until the process exits.
# Popen + thread-per-stream prints every log line as it is written,
# so you see iter/loss/N=[DENSIFY] in real-time inside the Colab cell.

proc = subprocess.Popen(
    cmd,
    env=env,
    cwd=str(REPO_DIR),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,   # merge stderr into stdout so order is preserved
    bufsize=1,                  # line-buffered
    text=True,
)

print("Training started — streaming output below.")
print("Watch for:")
print("  iter XXXX/12000  loss=0.XXXX  N=XX,XXX")
print("  [DENSIFY] iter=600  before=20000  after=22000+")
print("  [Trainer] Preview saved: .../preview_000250.png")
print("-" * 60)

for line in proc.stdout:
    print(line, end="", flush=True)

proc.wait()
if proc.returncode != 0:
    raise RuntimeError(f"Training failed (exit {proc.returncode})")

print("-" * 60)
print("\n✅ Training complete")


## Cell 8b — View Training Previews

Run this any time during or after training to see the latest preview renders.

In [ ]:
from pathlib import Path
from IPython.display import display, Image as IPImage
import ipywidgets as widgets

preview_dir = Path("/content/output/previews")

if not preview_dir.exists() or not list(preview_dir.glob("*.png")):
    print("No previews yet — training saves one every 250 iterations.")
    print("Re-run this cell after the first few hundred iterations.")
else:
    previews = sorted(preview_dir.glob("preview_*.png"))
    print(f"{len(previews)} preview(s) available — showing latest 4:\n")

    for p in previews[-4:]:
        iter_num = int(p.stem.replace('preview_', ''))
        print(f"  Iteration {iter_num:,}")
        display(IPImage(filename=str(p), width=480))
        print()


## Cell 9 — Validate Outputs

You should see `final.splat`, `final.ply`, and at least one `checkpoint_*.ckpt`.

In [ ]:
from pathlib import Path

for f in sorted(Path("/content/output").rglob("*")):
    if f.is_file():
        print(f"  {f.relative_to('/content/output')}  ({f.stat().st_size / 1e6:.1f} MB)")


## Cell 10 — Save to Drive

In [ ]:
import shutil

DRIVE_DST = "/content/drive/MyDrive/MonoSplatResults"
shutil.copytree("/content/output", DRIVE_DST, dirs_exist_ok=True)
print("✅ Saved to", DRIVE_DST)


## Cell 11 — Download SPLAT

In [ ]:
from google.colab import files

files.download("/content/output/exports/final.splat")
